# STIC fonctions validation

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime as dt

import matplotlib.pyplot as plt
import numpy as np
from pyproj import CRS

from sweat.common.solar import convert_to_local_time
from sweat.stic import functions, flux, smwetness

## Tetens' formula

In [ ]:
def tetens(t: float) -> float:
    a_tetens = 6.1078
    b_tetens = 17.27
    c_tetens = 237.3
    return a_tetens * np.exp((b_tetens * t) / (t + c_tetens))


def tetens_list(t: float) -> float:
    a_tetens = 6.13753  # In Tetens equation 6.1078
    b_tetens = 17.27
    c_tetens = 237.3
    return a_tetens * np.exp((b_tetens * t) / (t + c_tetens))


In [ ]:
ts = np.linspace(0, 50, num=20)
ea = tetens(ts)
ea2 = tetens_list(ts)

In [ ]:
plt.plot(ts, ea)
plt.plot(ts, ea2)

In [ ]:
plt.plot(ts, ea2 - ea)

In [ ]:
def slope(t):
    b_tetens = 17.27
    c_tetens = 237.3
    ea = tetens(ts)
    return b_tetens * c_tetens * ea / (ts + c_tetens) ** 2


def approx(t):
    return (
        45.03 + 3.014 * t + 0.05345 * t**2 + 0.00224 * t**3
    ) * 1e-2  # slope of SVP at TD (hpa/K)


In [ ]:
ts = np.linspace(0, 50, num=20)
s1 = slope(ts)
s2 = approx(ts)

In [ ]:
plt.plot(ts, s1)
plt.plot(ts, s2)

## Convert to local time

In [ ]:
utc_dt = dt.datetime(2023, 6, 16, 12, 0, 0, tzinfo=dt.timezone.utc)
lat = 40.0
lon = -74.0

In [ ]:
functions.convert_to_local_time(utc_dt, lon, lat)

In [ ]:
convert_to_local_time(utc_dt, lon, lat)

In [ ]:
easting = 585360
northing = 4428236
epsg = 32618

In [ ]:
convert_to_local_time(utc_dt, easting, northing, crs=CRS(epsg))

In [ ]:
functions.convert_to_local_time(utc_dt, easting, northing, crs=CRS(epsg))


## Compute psychrometrics

In [ ]:
def stic_f_pSYCHROMETRICS(TS, TA, TD,RH, P, Rd, Rw, Cpw, Cpd, sigma):
    esstar = 6.13753 * np.exp(
        (17.27 * TS) / (TS + 237.3)
    )  # saturation vapor pressure at surface temperature, TS (unit hPa)
    eastar = 6.13753 * np.exp(
        (17.27 * TA) / (TA + 237.3)
    )  # saturation vapor pressure at air temperature, TA (unit hPa)
    ea = (RH / 100) * (eastar)  # actual vapor pressure of air (unit hPa)

    DA = eastar - ea  # vapor pressure deficit of air (hPa)
    slopeTA = (
        4098.171 * eastar / (TA + 237.3) ** 2
    )  # slope of saturation vapor pressure versus air temperature (hPa)
    # Not necessary
    #TD = (
    #    237.3 * np.log(ea / 6.13753) / (17.27 - np.log(ea / 6.13753))
    #)  # dewpoint temperature (deg C)
    #if TD < -30:
    #    TD = np.nan
    DPD = TA - TD  # dewpoint depression

    slopeTD = 4098.171 * 6.13753 * np.exp(
        (17.27 * TD) / (TD + 237.3)
    ) / (TD + 237.3) ** 2
    slopeTS = 4098.171 * 6.13753 * np.exp(
        (17.27 * TS) / (TS + 237.3)
    ) / (TS + 237.3) ** 2
    slopeTaTd = (eastar - ea) / (
        TA - TD
    )  # slope of saturation vapor pressure at dewpoint temperature

    qref = (0.622 * ea * 100) / (101325 - (0.378 * ea * 100))
    rhoD = 101325 / (Rd * (TA + 273.15))  # density of dry air
    rho = rhoD * ((1 + qref) / (1 + qref * (Rw / Rd)))  # density of air
    CP = qref * Cpw + (1 - qref) * Cpd  # specific heat of air

    gR = ((4 * sigma * (TA + 273.15) ** 3) / CP) * (
        (Rd * (TA + 273.15)) / P
    )  # radiative conductance (m/s)(momentarily not needed)

    dTS = TS - TA  # difference between TS and TA
    # TU computation (surface dewpoint temperature)
    #TODO: Check s11 = slopeTD
    s11=(
        45.03 + 3.014 * TD + 0.05345 * TD**2 + 0.00224 * TD**3
    ) * 1e-2  # slope of SVP at TD (hpa/K)
    s22 = (esstar - ea) / (TS - TD)
    #TODO: Check s33 = slopeTS
    s33 = (
        45.03 + 3.014 * TS + 0.05345 * TS**2 + 0.00224 * TS**3
    ) * 1e-2  # slope of SVP at TS (hpa/K)
    s44 = (eastar - ea) / (TA - TD)

    return (
        esstar,
        eastar,
        ea,
        DA,
        slopeTA,
        TD,
        DPD,
        slopeTD,
        slopeTaTd,
        rho,
        CP,
        gR,
        dTS,
        s11,
        s22,
        s33,
        s44,
    )


In [ ]:
(
    esstar,
    eastar,
    ea,
    DA,
    slopeTA,
    TD,
    DPD,
    slopeTD,
    slopeTaTd,
    rho,
    CP,
    gR,
    dTS,
    s11,
    s22,
    s33,
    s44,
) = stic_f_pSYCHROMETRICS(
    TS=36,
    TA=28,
    TD=15,
    RH=45.1,
    P=101.325,
    Rd=286.9,
    Rw=461.5,
    Cpw=1846,
    Cpd=1005,
    sigma=5.67e-8,
)
for i in [esstar, eastar, ea, DA, slopeTA, s11, s22, s33, s44, rho, CP]:
    print(f'{i:6f},')

In [ ]:
for i in functions.compute_psychrometrics(ts=15, ta=15, td=15, rh=100.0):
    print(f'{i:6f},')

### Compute soil moisture

In [ ]:
def stic_f_SoilMoisture_INITIALIZE(
    gamma,
    slope,
    TS,
    TA,
    TD,
    dTS,
    RN,
    Lnet,
    fc,
    DA,
    eastar,
    ea,
    esstar,
    s11,
    s22,
    s33,
    s44,
):
    """
    This function estimates the soil moisture availability
    (M or wetness, 0-1) based on thermal and meteorological
    information.
    However, this M will be treated as initial M,
    which will be later on estimated through iteration in the actual ET estimation loop to
    establish feedback between M and biophysical states
    """

    # TU computation (surface dewpoint temperature)
    # s11 = (
    #    45.03 + 3.014 * TD + 0.05345 * TD**2 + 0.00224 * TD**3
    # ) * 1e-2  # slope of SVP at TD (hpa/K)
    # s22 = (esstar - ea) / (TS - TD)
    # s33 = (
    #    45.03 + 3.014 * TS + 0.05345 * TS**2 + 0.00224 * TS**3
    # ) * 1e-2  # slope of SVP at TS (hpa/K)
    # s44 = (eastar - ea) / (TA - TD)

    # Surface dewpoint temperature (degC)
    T0D = (esstar - ea - s33 * TS + s11 * TD) / (s11 - s33)

    # Surface moisture (Msurf)
    Msurf = (s11 / s22) * (
        (T0D - TD) / (TS - TD)
    )  # Surface wetness  #######To be checked

    if Msurf > 1:
        Msurf = 0.9999
    if Msurf < 0:
        Msurf = 0.0001

    # Surface vapor pressure and deficit
    esurf = ea + Msurf * (esstar - ea)
    Dsurf = esurf - ea

    # Separating soil and canopy wetness to form a composite surface moisture
    Ms = Msurf
    Mcan = fc * Msurf
    Msoil = (1 - fc) * Msurf

    TdewIndex = (TS - T0D) / (
        TA - TD
    )  # TdewIndex > 1 signifies super dry condition
    Ep_PT = (1.26 * slope * RN) / (
        slope + gamma
    )  # Potential evaporation (Priestley-Taylor eqn.)
    # Surface wetness comes from the soil, vegetation contribution is neglegible
    if (fc <= 0.25) & (TdewIndex < 1):
        Ms = Msoil
        Mcan = 0

    if (fc <= 0.25) & (TA > 10) & (TD < 0) & (Lnet < -125):
        Ms = Msoil
        Mcan = 0

    # Root zone moisture (Mrz)
    Mrz = (gamma * s11 * (T0D - TD)) / (
        slope * s33 * (TS - TD)
        + gamma * s44 * (TA - TD)
        - slope * s11 * (T0D - TD)
    )

    if Mrz > 1:
        Mrz = 0.9999
    if Mrz < 0:
        Mrz = 0.0001

    # Combine M to account for Hysteresis and initial estimation of surface vapor pressure
    M = Ms
    if (Ep_PT > RN) & (dTS > 0):
        M = Mrz
    if (Ep_PT > RN) & (fc <= 0.25):
        M = Mrz
    if (Ep_PT > RN) & (Dsurf > DA):
        M = Mrz

    if (fc <= 0.25) & (dTS > 0) & (TA > 10) & (TD < 0) & (Lnet < -125):
        M = Mrz
    if (fc <= 0.25) & (dTS > 0) & (TA > 10) & (TD < 0) & (Dsurf > DA):
        M = Mrz
    if (Ep_PT < RN) & (fc <= 0.25) & (Dsurf > DA):
        M = Mrz

    es = ea + M * (esstar - ea)

    # vapor pressure deficit at surface
    Ds = esstar - es

    return (M, Mcan, Msoil, Ms, Mrz, s11, s22, s33, s44, es, T0D, Ds, TdewIndex)


In [ ]:
(M, Mcan, Msoil, Ms, Mrz, s11, s22, s33, s44, es, T0D, Ds, TdewIndex) = (
    stic_f_SoilMoisture_INITIALIZE(
        gamma=0.67,
        slope=3.0,
        TS=25,
        TA=10,
        TD=1,
        dTS=15,
        RN=250,
        Lnet=100,
        fc=0.7,
        DA=9.7,
        eastar=31.677,
        ea=21.971,
        esstar=42.43,
        s11=1.5,
        s22=1.860,
        s33=2.434,
        s44=1.618,
    )
)
# (M, Mcan, Msoil, Ms, Mrz, s11, s22, s33, s44, es, T0D, Ds, TdewIndex)
for i in [M, Mcan, Msoil, Ms, Mrz, es, T0D, Ds]:
    print(f'{i:6f},')

In [ ]:
for i in smwetness.initialize_soil_moisture(
    slope=3.0,
    ts=25,
    ta=10,
    td=1,
    rn=250,
    ln=100,
    fc=0.7,
    da=9.7,
    ea=21.971,
    esstar=42.43,
    s1=1.5,
    s2=1.860,
    s3=2.434,
    s4=1.618,
):
    print(f'{i:6f},')

In [ ]:
def stic_f_SoilMoisture_ITERATE(
    gamma,
    slope,
    s1,
    s2,
    s3,
    s4,
    TS,
    TA,
    dTS,
    TD,
    T0D,
    RN,
    Lnet,
    fc,
    DA,
    D0,
    ea,
    e0star,
    esstar,
):
    """
    This functions estimates the soil moisture availability (M) (or
    wetness) (value 0 to 1) based on thermal IR and meteorological
    information. However, this M will be treated as initial M, which will be
    later on estimated through iteration in the actual ET estimation loop to
    establish feedback between M and biophysical states
    """
    # Surface moisture (Msurf)
    k = (e0star - ea) / (esstar - ea)
    Msurf = (s1 / (k * s2)) * ((T0D - TD) / (TS - TD))  # Surface wetness

    if Msurf > 1:
        Msurf = 0.9999
    if Msurf < 0:
        Msurf = 0.0001

    # Separating soil and canopy wetness to form a composite surface moisture
    Ms = Msurf
    Mcan = fc * Msurf
    Msoil = (1 - fc) * Msurf

    TdewIndex = (TS - T0D) / (TA - TD)
    Ep_PT = (1.26 * slope * RN) / (
        slope + gamma
    )  # Potential evaporation (Priestley-Taylor eqn.)

    if (fc <= 0.25) & (TdewIndex < 1):
        Ms = Msoil
    if (fc <= 0.25) & (TA > 10) & (TD < 0) & (Lnet < -125):
        Ms = Msoil

    if (fc <= 0.25) & (TdewIndex < 1):
        Mcan = 0
    if (fc <= 0.25) & (TA > 10) & (TD < 0) & (Lnet < -125):
        Mcan = 0

    # Rootzone moisture (Mrz)
    Mrz = (gamma * s1 * (T0D - TD)) / (
        slope * s3 * (TS - TD)
        + gamma * s4 * (TA - TD)
        - slope * s1 * (T0D - TD)
    )

    if Mrz > 1:
        Mrz = 0.9999
    if Mrz < 0:
        Mrz = 0.0001

    TdewIndex = (TS - T0D) / (TA - TD)
    Ep_PT = (1.26 * slope * RN) / (
        slope + gamma
    )  # Potential evaporation (Priestley-Taylor eq.)
    # Combine M to account for Hysteresis and initial estimation of surface vapor pressure
    M = Ms
    if (Ep_PT > RN) & (dTS > 0) & (fc <= 0.25) & (D0 > DA) & (TdewIndex < 1):
        M = Mrz
    if (
        (Lnet < -125)
        & (dTS > 0)
        & (fc <= 0.25)
        & (D0 > DA)
        & (TA > 10)
        & (TD < 0)
    ):
        M = Mrz

    return (M, Ms, Mcan, Msoil, Mrz)

In [ ]:
(M, Ms, Mcan, Msoil, Mrz) = stic_f_SoilMoisture_ITERATE(
    gamma=0.67,
    slope=4.0,
    s1=1.5,
    s2=1.8,
    s3=2.4,
    s4=1.6,
    TS=26,
    TA=25,
    dTS=7,
    TD=-1,
    T0D=20,
    RN=150,
    Lnet=100,
    fc=0.2,
    DA=0.05,
    D0=0.2,
    ea=21.5,
    e0star=33.5,
    esstar=43.67,
)
for i in [M, Ms, Mcan, Msoil, Mrz]:
    print(f'{i:6f},')

In [ ]:
(M, Ms, Mcan, Msoil, Mrz) = smwetness.iterate_soil_moisture(
    slope=4.0,
    s1=1.5,
    s2=1.8,
    s3=2.4,
    s4=1.6,
    ts=26,
    ta=25,
    delta_t=7,
    td=-1,
    t0d=20,
    rn=150,
    ln=100,
    fc=0.2,
    da=0.05,
    d0=0.2,
    ea=21.5,
    e0star=33.5,
    esstar=43.67,
)
for i in [M, Ms, Mcan, Msoil, Mrz]:
    print(f'{i:6f},')

### Compute G flux

In [ ]:
def stic_f_G_PHI_actualsurface(RN, lai, ttSEC, M):
    KRN = 0.6
    RNsoil = RN * np.exp(-KRN * lai)
    cgMIN = 0.05  # for wet surface
    cgMAX = 0.35  # for dry surface, in water-controlled ecosystems
    #     cgMAX = 0.20 # for dry surface, in radiation-controlled ecosystems

    tgMIN = 74000  # for wet surface
    tgMAX = 100000  # for dry surface

    solNooN = 12 * 60 * 60
    tg0 = solNooN - ttSEC

    # Estimating GHF according to Santanello and Friedl (2003)
    cg = (1 - M) * cgMAX + M * cgMIN
    tg = (1 - M) * tgMAX + M * tgMIN

    # G = RN*cg*np.cos(2*np.pi*(tg0 + 10800)/tg)
    # G[RN < 0] = -G[RN < 0]

    G = (
        RNsoil * cg * np.cos(2 * np.pi * (tg0 + 10800) / tg)
    )  # Correction on 12/11/2021
    if RNsoil < 0:
        G = -G

    PHI = RN - G

    return (cg, tg, tg0, G, PHI)

In [ ]:
(cg, tg, tg0, G, PHI) = stic_f_G_PHI_actualsurface(
    RN=-50, lai=2, ttSEC=36000, M=0.6
)
print(G)


In [ ]:
flux.compute_g_flux(rn=-50, lai=2, local_time=36000, m=0.6)

### Compute state equations

In [ ]:
def stic_f_StateEQ(rho, CP, gamma, alfa, slope, PHI, e0, ea, e0star, M):
    # Aerodynamic conductance
    gB = (2 * PHI * alfa * slope * gamma) / (
        2 * CP * slope * e0 * rho
        - 2 * CP * slope * ea * rho
        - 2 * CP * ea * gamma * rho
        + CP * e0 * gamma * rho
        + CP * e0star * gamma * rho
        - CP * M * e0 * gamma * rho
        + CP * M * e0star * gamma * rho
    )

    # Surface conductance
    denominator = (
        CP * e0star**2 * gamma * rho
        - CP * e0**2 * gamma * rho
        - 2 * CP * slope * e0**2 * rho
        + 2 * CP * slope * ea * e0 * rho
        - 2 * CP * slope * ea * e0star * rho
        + 2 * CP * slope * e0 * e0star * rho
        + 2 * CP * ea * e0 * gamma * rho
        - 2 * CP * ea * e0star * gamma * rho
        + CP * M * e0**2 * gamma * rho
        + CP * M * e0star**2 * gamma * rho
        - 2 * CP * M * e0 * e0star * gamma * rho
    )
    if abs(denominator) < 1.0e-10:
        denominator = 0.00001  # when e0star == e0
    gS = (
        -(
            2
            * (
                PHI * alfa * slope * ea * gamma
                - PHI * alfa * slope * e0 * gamma
            )
        )
        / denominator
    )

    # T0 - TA
    dT = (
        2 * slope * e0
        - 2 * slope * ea
        - 2 * ea * gamma
        + e0 * gamma
        + e0star * gamma
        - M * e0 * gamma
        + M * e0star * gamma
        + 2 * alfa * slope * ea
        - 2 * alfa * slope * e0
    ) / (2 * alfa * slope * gamma)

    # Evaporative fraction
    EF = -(2 * alfa * slope * ea - 2 * alfa * slope * e0) / (
        2 * slope * e0
        - 2 * slope * ea
        - 2 * ea * gamma
        + e0 * gamma
        + e0star * gamma
        - M * e0 * gamma
        + M * e0star * gamma
    )

    # Adjust the abnormal conductances
    if gB < 0:
        gB = 0.0001
    if gB > 0.2:
        gB = 0.2
    if gS < 0:
        gS = 0.0001
    if gS > 0.06:
        gS = 0.06

    # Maximum surface-air temperature difference rarely overpasses 20 degC
    if dT < -10:
        dT = -10
    if dT > 20:
        dT = 20
    if EF < 0:
        EF = 0.0001
    if EF > 1:
        EF = 1

    return (gB, gS, dT, EF)

In [ ]:
(gB, gS, dT, EF) = stic_f_StateEQ(
    rho=0.04,
    CP=1026,
    gamma=0.67,
    alfa=1.3,
    slope=2.0,
    PHI=50,
    e0=32.5,
    ea=45.7,
    e0star=32.5,
    M=0.7,
)
print(gB, gS, dT, EF)

In [ ]:
(gB, gS, dT, EF) = functions.compute_state_equations(
    rho=0.04,
    cp=1026,
    alpha=1.3,
    slope=2.0,
    phi=50,
    e0=32.5,
    ea=32.5,
    e0star=32.5,
    m=0.7,
)
print(gB, gS, dT, EF)
